In [7]:
import re

ord("A")

65

In [8]:
ord("👋")

128075

In [10]:
[ord(x) for x in "你好 👋 (hello in chinese)"]

[20320,
 22909,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 99,
 104,
 105,
 110,
 101,
 115,
 101,
 41]

In [11]:
"你好 👋 (hello in chinese)".encode("utf-8")

b'\xe4\xbd\xa0\xe5\xa5\xbd \xf0\x9f\x91\x8b (hello in chinese)'

In [13]:
list("你好 👋 (hello in chinese)".encode("utf-32"))

[255,
 254,
 0,
 0,
 96,
 79,
 0,
 0,
 125,
 89,
 0,
 0,
 32,
 0,
 0,
 0,
 75,
 244,
 1,
 0,
 32,
 0,
 0,
 0,
 40,
 0,
 0,
 0,
 104,
 0,
 0,
 0,
 101,
 0,
 0,
 0,
 108,
 0,
 0,
 0,
 108,
 0,
 0,
 0,
 111,
 0,
 0,
 0,
 32,
 0,
 0,
 0,
 105,
 0,
 0,
 0,
 110,
 0,
 0,
 0,
 32,
 0,
 0,
 0,
 99,
 0,
 0,
 0,
 104,
 0,
 0,
 0,
 105,
 0,
 0,
 0,
 110,
 0,
 0,
 0,
 101,
 0,
 0,
 0,
 115,
 0,
 0,
 0,
 101,
 0,
 0,
 0,
 41,
 0,
 0,
 0]

In [19]:
tokens = "你好 👋 (hello in chinese)"
tokens = tokens.encode("utf-8")
tokens = list(map(int, tokens))
tokens

[228,
 189,
 160,
 229,
 165,
 189,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 99,
 104,
 105,
 110,
 101,
 115,
 101,
 41]

In [20]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
print(stats)

{(228, 189): 1, (189, 160): 1, (160, 229): 1, (229, 165): 1, (165, 189): 1, (189, 32): 1, (32, 240): 1, (240, 159): 1, (159, 145): 1, (145, 139): 1, (139, 32): 1, (32, 40): 1, (40, 104): 1, (104, 101): 1, (101, 108): 1, (108, 108): 1, (108, 111): 1, (111, 32): 1, (32, 105): 1, (105, 110): 2, (110, 32): 1, (32, 99): 1, (99, 104): 1, (104, 105): 1, (110, 101): 1, (101, 115): 1, (115, 101): 1, (101, 41): 1}


In [23]:
print(sorted(((v,k) for k,v in stats.items()), reverse=True))

[(2, (105, 110)), (1, (240, 159)), (1, (229, 165)), (1, (228, 189)), (1, (189, 160)), (1, (189, 32)), (1, (165, 189)), (1, (160, 229)), (1, (159, 145)), (1, (145, 139)), (1, (139, 32)), (1, (115, 101)), (1, (111, 32)), (1, (110, 101)), (1, (110, 32)), (1, (108, 111)), (1, (108, 108)), (1, (104, 105)), (1, (104, 101)), (1, (101, 115)), (1, (101, 108)), (1, (101, 41)), (1, (99, 104)), (1, (40, 104)), (1, (32, 240)), (1, (32, 105)), (1, (32, 99)), (1, (32, 40))]


In [24]:
chr(101), chr(32)

('e', ' ')

In [25]:
top_pair = max(stats, key=stats.get)
print(top_pair)

(105, 110)


In [27]:
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

[5, 6, 99, 9, 1]


In [28]:
tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("length:", len(tokens2))

[228, 189, 160, 229, 165, 189, 32, 240, 159, 145, 139, 32, 40, 104, 101, 108, 108, 111, 32, 256, 32, 99, 104, 256, 101, 115, 101, 41]
length: 28


In [30]:
# making the training text longer to have more representative token statistics
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = """Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."""
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0-255

In [35]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# ---
vocab_size = 276 # the desired final vocabulary size 
num_merges = vocab_size - 256 # how many merges we need to do
ids = list(tokens) # copy so we don't destory the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    
    print(f"merging {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (101, 32) into a new token 256
merging (240, 159) into a new token 257
merging (226, 128) into a new token 258
merging (105, 110) into a new token 259
merging (115, 32) into a new token 260
merging (97, 110) into a new token 261
merging (116, 104) into a new token 262
merging (257, 133) into a new token 263
merging (257, 135) into a new token 264
merging (97, 114) into a new token 265
merging (239, 189) into a new token 266
merging (258, 140) into a new token 267
merging (267, 264) into a new token 268
merging (101, 114) into a new token 269
merging (111, 114) into a new token 270
merging (116, 32) into a new token 271
merging (259, 103) into a new token 272
merging (115, 116) into a new token 273
merging (261, 100) into a new token 274
merging (32, 262) into a new token 275


In [36]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 616
ids length: 451
compression ratio: 1.37X


In [37]:
# decoding
vocab = {idx: bytes([idx]) for idx in range(256)} # start with the raw byte tokens
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    
def decode(ids):
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8")
    return text

print(decode([97]))

a


In [38]:
print(decode([128]))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

In [39]:
def decode(ids):
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

print(decode([128]))

�


In [40]:
merges

{(101, 32): 256,
 (240, 159): 257,
 (226, 128): 258,
 (105, 110): 259,
 (115, 32): 260,
 (97, 110): 261,
 (116, 104): 262,
 (257, 133): 263,
 (257, 135): 264,
 (97, 114): 265,
 (239, 189): 266,
 (258, 140): 267,
 (267, 264): 268,
 (101, 114): 269,
 (111, 114): 270,
 (116, 32): 271,
 (259, 103): 272,
 (115, 116): 273,
 (261, 100): 274,
 (32, 262): 275}

In [41]:
stats

{(239, 188): 1,
 (188, 181): 1,
 (181, 266): 1,
 (266, 142): 1,
 (142, 266): 1,
 (266, 137): 1,
 (137, 266): 1,
 (266, 131): 1,
 (131, 266): 1,
 (266, 143): 1,
 (143, 266): 1,
 (266, 132): 1,
 (132, 266): 1,
 (266, 133): 1,
 (133, 33): 1,
 (33, 32): 2,
 (32, 263): 1,
 (263, 164): 1,
 (164, 263): 1,
 (263, 157): 1,
 (157, 263): 1,
 (263, 152): 1,
 (152, 263): 1,
 (263, 146): 1,
 (146, 263): 1,
 (263, 158): 1,
 (158, 263): 1,
 (263, 147): 1,
 (147, 263): 1,
 (263, 148): 1,
 (148, 258): 1,
 (258, 189): 1,
 (189, 32): 1,
 (32, 264): 1,
 (264, 186): 1,
 (186, 268): 1,
 (268, 179): 1,
 (179, 268): 1,
 (268, 174): 1,
 (174, 268): 1,
 (268, 168): 1,
 (168, 268): 1,
 (268, 180): 1,
 (180, 268): 1,
 (268, 169): 1,
 (169, 268): 1,
 (268, 170): 1,
 (170, 33): 1,
 (32, 257): 1,
 (257, 152): 1,
 (152, 132): 1,
 (132, 32): 1,
 (32, 84): 1,
 (84, 104): 1,
 (104, 256): 1,
 (256, 118): 1,
 (118, 269): 2,
 (269, 121): 1,
 (121, 32): 2,
 (32, 110): 2,
 (110, 97): 1,
 (97, 109): 4,
 (109, 256): 2,
 (256, 2

In [42]:
# encode
def encode(text):
    # given a Python string, return list of integers (the tokens)
    tokens = list(text.encode("utf-8"))
    while True:
        stats = get_stats(tokens)
        pair = min(stats, key = lambda p: merges.get(p, float("inf"))) 
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode("hello world"))

[104, 101, 108, 108, 111, 32, 119, 270, 108, 100]


In [43]:
print(encode("h"))

ValueError: min() arg is an empty sequence

In [44]:
print(encode("he"))

[104, 101]


In [45]:
# encode
def encode(text):
    # given a Python string, return list of integers (the tokens)
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key = lambda p: merges.get(p, float("inf"))) 
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode("h"))

[104]


In [46]:
print(decode(encode("hello world")))

hello world


In [47]:
text2 = decode(encode(text))
print(text2 == text)

True


In [48]:
valtext = "Most programming languages have libraries available to handle the gory low-level details of text manipulation, but as a programmer, you’ll still need to know about certain Unicode features in order to know when and how to apply them."
valtext2 = decode(encode(valtext))
print(valtext2 == valtext)

True


In [52]:
# forced splits using regex patterns (GPT series)
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
print(re.findall(gpt2pat, "Hello, world"))

['Hello', ',', ' world']


In [53]:
print(re.findall(gpt2pat, "Hello world how are you"))

['Hello', ' world', ' how', ' are', ' you']


In [54]:
print(re.findall(gpt2pat, "Hello world123 how are you"))

['Hello', ' world', '123', ' how', ' are', ' you']


In [57]:
print(re.findall(gpt2pat, "Hello've world123 how's are      you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', '     ', ' you', '!!!?']


In [58]:
example = """
for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
"""
print(re.findall(gpt2pat, example))

['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', ':', '\n       ', ' print', '(', 'i', ')', '\n']


In [59]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    Hello, world!!!"))

[220, 220, 220, 18435, 11, 995, 10185]


In [60]:
# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    Hello, world!!!"))

[262, 22691, 11, 1917, 12340]


In [63]:
# download these two files
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

--2026-05-05 15:17:36--  https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.209.18.33
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.209.18.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 456318 (446K) [application/octet-stream]
Saving to: ‘vocab.bpe’

vocab.bpe           100%[===================>] 445.62K   673KB/s    in 0.7s    

2026-05-05 15:17:38 (673 KB/s) - ‘vocab.bpe’ saved [456318/456318]

--2026-05-05 15:17:38--  https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.209.18.33
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.209.18.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1042301 (1018K) [application/json]
Saving to: ‘